## 1. Import thư viện

In [2]:
import re
import math
import pickle
import numpy as np
import pandas as pd

from collections import Counter
from underthesea import word_tokenize

## 2. Load TF-IDF index, BM25 index và metadata

In [3]:
with open("../models/tfidf_index.pkl", "rb") as f:
    tfidf_index = pickle.load(f)

with open("../models/bm25_index.pkl", "rb") as f:
    bm25_index = pickle.load(f)

df = pd.read_pickle("../models/metadata.pkl")

idf = tfidf_index["idf"]
tfidf_docs = tfidf_index["tfidf_docs"]

bm25_idf = bm25_index["bm25_idf"]
doc_term_freqs = bm25_index["doc_term_freqs"]
doc_lengths = bm25_index["doc_lengths"]
avg_doc_length = bm25_index["avg_doc_length"]
N = bm25_index["N"]

print("Metadata shape:", df.shape)
print("Number of documents:", N)

df.head()

Metadata shape: (182566, 6)
Number of documents: 182566


,id,title,topic,source,url,processed_text
0,218270,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,docbao.vn,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...,tên cướp tiệm vàng huế đại_úy công_an công_tác...
1,218269,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,vtc.vn,https://vtc.vn/bo-qua-mang-5g-nga-tien-thang-t...,bỏ_qua mạng 5 g nga tiến thẳng 4 g lên 6 g bỏ_...
2,218268,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,thanhnien.vn,https://thanhnien.vn/dia-phuong-nao-dung-dau-c...,địa_phương nào đứng đầu cả nước tổng_điểm 3 mô...
3,218267,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,vnexpress,https://vnexpress.net/nguoi-chet-trong-mua-lu-...,người chết mưa_lũ nghìn mỹ tăng lên 28 người c...
4,218266,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,soha,https://soha.vn/hai-phong-hinh-anh-xe-dien-gay...,hải_phòng hình_ảnh xe điên gây tai_nạn liên_ho...


## 3. Hàm tiền xử lý query

In [4]:
vietnamese_stopwords = set([
    "là", "và", "của", "có", "cho", "với", "trong", "khi",
    "những", "các", "một", "được", "đã", "đang", "này",
    "đó", "thì", "mà", "ở", "về", "từ", "đến", "theo",
    "sau", "trước", "trên", "dưới", "vào", "ra", "năm",
    "ngày", "tháng", "cũng", "như", "nếu", "vì", "do",
    "để", "bị", "tại", "nên", "sẽ", "rằng", "nhiều"
])

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-ZÀ-ỹ0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def remove_stopwords(text):
    tokens = text.split()
    tokens = [token for token in tokens if token not in vietnamese_stopwords]
    return " ".join(tokens)

def preprocess(text):
    text = clean_text(text)
    text = word_tokenize(text, format="text")
    text = remove_stopwords(text)
    return text

## 4. Tính điểm TF-IDF cho query

In [5]:
def get_query_tfidf(query):
    processed_query = preprocess(query)
    query_tokens = processed_query.split()

    token_counts = Counter(query_tokens)
    total_tokens = len(query_tokens)

    query_tfidf = {}

    if total_tokens > 0:
        for token, count in token_counts.items():
            if token in idf:
                tf = count / total_tokens
                query_tfidf[token] = tf * idf[token]

    return query_tfidf

## 5. Tính cosine similarity thủ công

In [6]:
def cosine_dict(vec1, vec2):
    common_terms = set(vec1.keys()) & set(vec2.keys())

    numerator = sum(vec1[t] * vec2[t] for t in common_terms)

    norm1 = math.sqrt(sum(v ** 2 for v in vec1.values()))
    norm2 = math.sqrt(sum(v ** 2 for v in vec2.values()))

    if norm1 == 0 or norm2 == 0:
        return 0.0

    return numerator / (norm1 * norm2)

## 6. Lấy ranking từ TF-IDF

In [7]:
def get_tfidf_ranking(query):
    query_vec = get_query_tfidf(query)

    scores = []

    for doc_vec in tfidf_docs:
        score = cosine_dict(query_vec, doc_vec)
        scores.append(score)

    scores = np.array(scores)
    ranked_indices = scores.argsort()[::-1]

    return ranked_indices, scores

## 7. Tính điểm BM25 cho query

In [8]:
def get_bm25_scores(query, k1=1.5, b=0.75):
    processed_query = preprocess(query)
    query_tokens = processed_query.split()

    scores = []

    for doc_idx in range(N):
        score = 0.0

        doc_tf = doc_term_freqs[doc_idx]
        doc_len = doc_lengths[doc_idx]

        for token in query_tokens:
            if token not in bm25_idf:
                continue

            tf = doc_tf.get(token, 0)

            numerator = tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * doc_len / avg_doc_length)

            score += bm25_idf[token] * numerator / denominator

        scores.append(score)

    return np.array(scores)

## 8. Lấy ranking từ BM25

In [9]:
def get_bm25_ranking(query, k1=1.5, b=0.75):
    scores = get_bm25_scores(query, k1=k1, b=b)
    ranked_indices = scores.argsort()[::-1]

    return ranked_indices, scores

## 9. Hybrid Search bằng Reciprocal Rank Fusion

RRF không cộng trực tiếp điểm TF-IDF và BM25.

Thay vào đó, RRF kết hợp dựa trên thứ hạng của document trong từng danh sách kết quả.

Công thức:

RRF(d) = 1 / (k + rank_TFIDF(d)) + 1 / (k + rank_BM25(d))

Trong đó:
- rank_TFIDF(d): thứ hạng của document theo TF-IDF
- rank_BM25(d): thứ hạng của document theo BM25
- k: hệ số làm mượt, thường dùng 60

In [11]:
def search_hybrid_rrf(query, top_k=10, rrf_k=60):
    tfidf_ranked, tfidf_scores = get_tfidf_ranking(query)
    bm25_ranked, bm25_scores = get_bm25_ranking(query)

    rrf_scores = np.zeros(N)

    for rank, doc_idx in enumerate(tfidf_ranked):
        rrf_scores[doc_idx] += 1 / (rrf_k + rank + 1)

    for rank, doc_idx in enumerate(bm25_ranked):
        rrf_scores[doc_idx] += 1 / (rrf_k + rank + 1)

    top_indices = rrf_scores.argsort()[::-1][:top_k]

    results = df.iloc[top_indices].copy()

    results["tfidf_score"] = tfidf_scores[top_indices]
    results["bm25_score"] = bm25_scores[top_indices]
    results["rrf_score"] = rrf_scores[top_indices]

    return results[[
        "id", "title", "topic", "source", "url",
        "tfidf_score", "bm25_score", "rrf_score"
    ]]

## 10. Test Hybrid RRF

In [12]:
search_hybrid_rrf("giá xăng tăng", top_k=10)

,id,title,topic,source,url,tfidf_score,bm25_score,rrf_score
162668,21755,Ngày mai (21-6): Dự báo giá xăng tiếp tục lập ...,Kinh doanh,anninhthudo,https://www.anninhthudo.vn/ngay-mai-21-6-du-ba...,0.649001,16.373553,0.031281
122500,68195,"Giá xăng sẽ giảm vào chiều nay, 1-7?",Kinh tế,qdnd.vn,https://www.qdnd.vn/kinh-te/tin-tuc/gia-xang-s...,0.630754,16.462675,0.030886
127838,60277,Giá xăng có thể giảm vào ngày mai | VTV.VN,,vtv.vn,https://vtv.vn/kinh-te/gia-xang-co-the-giam-va...,0.630407,16.365314,0.030159
176767,6340,Người Việt ở Mỹ thắt chặt chi tiêu khi giá xăn...,Thế giới,vnexpress,https://vnexpress.net/nguoi-viet-o-my-that-cha...,0.677436,16.261647,0.029710
39441,169437,"Giá xăng giảm kỷ lục, giá hàng hóa vẫn ""đứng t...",Kinh tế,danviet,https://danviet.vn/gia-xang-giam-ky-luc-gia-ha...,0.638133,16.304410,0.029412
174038,9349,"Xăng tăng giá kỷ lục, bạn sẽ sống sao?",Giới trẻ,thanhnien.vn,https://thanhnien.vn/xang-tang-gia-ky-luc-ban-...,0.677098,16.242885,0.028898
127637,60540,Giá xăng giảm sau 7 lần tăng mạnh liên tiếp,Kinh Doanh,vietnamnet.vn,https://vietnamnet.vn/gia-xang-giam-sau-7-lan-...,0.616245,16.356713,0.028718
148199,36811,Lo ngại giá xăng trong nước tăng tiếp vì điều này,Sản xuất - Tiêu dùng,24h.com.vn,https://www.24h.com.vn/san-xuat-tieu-dung/lo-n...,0.626361,16.289004,0.028175
126629,61828,Giá xăng có thể giảm vào ngày mai,,soha,https://soha.vn/gia-xang-co-the-giam-vao-ngay-...,0.594481,16.359113,0.027673
127121,61150,Giá xăng có thể giảm vào ngày mai,Xã hội,cafebiz,https://cafebiz.vn/gia-xang-co-the-giam-vao-ng...,0.593368,16.351695,0.026916
